# Sistema Massa-Mola

Vamos resolver a equação do sistema massa-mola com 4 métodos. Em cada bloco, o código calcula a solução, mostra alguns valores e plota `x(t)`.


Equação do problema:

`x'' + (k/m)x = 0`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import odeint
try:
    from IPython.display import HTML
except Exception:
    HTML = None

m, k = 1.0, 1.0                                # parâmetros do sistema
x0, v0 = 1.0, 0.0                              # condição inicial
dt = 0.02                                      # passo de tempo
t = np.arange(0.0, 10.0 + dt, dt)             # pontos de tempo
N = len(t)                                     # número de pontos

def mostrar(nome, y):
    print(f"\nSaída numérica - {nome}")
    print(" i   t(s)      x(m)")
    for i in range(10):
        print(f"{i:2d}  {t[i]:6.3f}   {y[i]:10.6f}")
    plt.figure(figsize=(8, 4))
    plt.plot(t, y, lw=2)
    plt.xlabel('tempo (s)')
    plt.ylabel('x(t)')
    plt.title(nome)
    plt.grid(alpha=0.3)
    plt.show()


## Método 1: Euler


In [ ]:
def euler():
    x = np.zeros(N)                            # posição
    v = np.zeros(N)                            # velocidade
    x[0], v[0] = x0, v0                        # condição inicial
    for i in range(N - 1):
        a = -(k / m) * x[i]                    # aceleração
        x[i + 1] = x[i] + v[i] * dt            # novo x
        v[i + 1] = v[i] + a * dt               # nova v
    return x, v

x_eu, v_eu = euler()
mostrar('Euler', x_eu)


## Método 2: Euler-Cromer


In [ ]:
def euler_cromer():
    x = np.zeros(N)                            # posição
    v = np.zeros(N)                            # velocidade
    x[0], v[0] = x0, v0                        # condição inicial
    for i in range(N - 1):
        a = -(k / m) * x[i]                    # aceleração
        v[i + 1] = v[i] + a * dt               # nova v
        x[i + 1] = x[i] + v[i + 1] * dt        # novo x
    return x, v

x_ec, v_ec = euler_cromer()
mostrar('Euler-Cromer', x_ec)


## Método 3: RK4


In [ ]:
def derivadas(x, v):
    return v, -(k / m) * x                     # x' e v'

def rk4():
    x = np.zeros(N)                            # posição
    v = np.zeros(N)                            # velocidade
    x[0], v[0] = x0, v0                        # condição inicial
    for i in range(N - 1):
        k1x, k1v = derivadas(x[i], v[i])
        k2x, k2v = derivadas(x[i] + 0.5*dt*k1x, v[i] + 0.5*dt*k1v)
        k3x, k3v = derivadas(x[i] + 0.5*dt*k2x, v[i] + 0.5*dt*k2v)
        k4x, k4v = derivadas(x[i] + dt*k3x, v[i] + dt*k3v)
        x[i + 1] = x[i] + (dt/6)*(k1x + 2*k2x + 2*k3x + k4x)
        v[i + 1] = v[i] + (dt/6)*(k1v + 2*k2v + 2*k3v + k4v)
    return x, v

x_rk, v_rk = rk4()
mostrar('RK4', x_rk)


## Método 4: odeint


In [ ]:
def sistema(y, tempo):
    x, v = y                                   # separa o estado
    return [v, -(k / m) * x]                   # devolve as derivadas

sol = odeint(sistema, [x0, v0], t)            # integração pronta
x_od, v_od = sol[:, 0], sol[:, 1]             # separa x e v
mostrar('odeint', x_od)


## Bônus visual

A animação fica separada da parte principal do notebook.


In [ ]:
def indices_animacao(total, max_frames=160):
    return np.unique(np.linspace(0, total - 1, min(max_frames, total), dtype=int))

def desenhar_mola(xm, y0, xw=-1.2, esp=12, amp=0.05, pts=120):
    xf = max(xm - 0.08, xw + 0.05)             # fim da mola
    xs = np.linspace(xw, xf, pts)              # pontos em x
    ys = y0 + amp*np.sin(np.linspace(0, 2*np.pi*esp, pts))
    ys[0], ys[-1] = y0, y0                     # pontas retas
    return xs, ys

def animar(x):
    fr = indices_animacao(N)                   # menos frames
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
    ax1.set(xlim=(-1.45, 1.45), ylim=(-0.35, 0.35), xlabel='x (m)', ylabel='y', title='Massa-mola')
    ax1.grid(alpha=0.3)
    ax1.axvline(-1.2, color='gray', lw=3)
    ax2.set(xlim=(0, t[-1]), ylim=(1.1*x.min(), 1.1*x.max()), xlabel='tempo (s)', ylabel='x(t)', title='odeint')
    ax2.grid(alpha=0.3)
    mola, = ax1.plot([], [], color='tab:green', lw=2.5)
    massa, = ax1.plot([], [], 'o', color='tab:green', ms=12)
    curva, = ax2.plot([], [], color='tab:green', lw=2)
    def update(i):
        xs, ys = desenhar_mola(x[i], 0.0)      # desenha a mola
        mola.set_data(xs, ys)
        massa.set_data([x[i]], [0.0])
        curva.set_data(t[:i + 1], x[:i + 1])
        return mola, massa, curva
    ani = FuncAnimation(fig, update, frames=fr, interval=20, blit=False)
    fig._ani = ani
    plt.tight_layout()
    return ani

ani = animar(x_od)
html = HTML(ani.to_jshtml())
plt.close(ani._fig)
html
